# Unidad 3 - RAG Agéntico: Alfred, el Anfitrión Perfecto

En esta unidad construiremos un agente completo que ayuda a **Alfred** a organizar la gala más extravagante del siglo, usando **Retrieval-Augmented Generation (RAG) Agéntico**.

**Temas cubiertos:**
1. Conceptos de RAG y RAG Agéntico
2. Herramientas auxiliares: búsqueda web, clima y estadísticas del Hub
3. Herramienta RAG personalizada para información de invitados (BM25)
4. Ensamblaje del agente Alfred con todas las herramientas
5. Ejemplos de uso y conversación con memoria

**Framework principal:** LlamaIndex (con Ollama local — `qwen2:7b`)

---
📖 Referencia: [HuggingFace Agents Course - Unit 3](https://huggingface.co/learn/agents-course/en/unit3/agentic-rag/agent?agents-frameworks=llama-index)

## 1. Conceptos: RAG y RAG Agéntico

Los LLMs se entrenan sobre grandes volúmenes de datos para aprender conocimiento general. Sin embargo, **su conocimiento puede no estar actualizado** o puede no incluir información privada/específica de tu dominio.

**RAG (Retrieval-Augmented Generation)** resuelve esto recuperando información relevante de tus datos y pasándosela al LLM como contexto.

![RAG](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/unit2/llama-index/rag.png)

**RAG Agéntico** va un paso más allá: en lugar de recuperar automáticamente contexto de documentos, el **agente decide cuándo y qué herramienta usar** para responder la pregunta.

![Agentic RAG](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/unit2/llama-index/agentic-rag.png)

### Los requisitos de la Gala

Alfred, nuestro agente, necesita:
- **Conocimiento de deportes, cultura y ciencia** (búsqueda web)
- **Información sobre los invitados** (RAG sobre dataset de invitados)
- **Estado del clima** (herramienta meteorológica para los fuegos artificiales)
- **Información sobre creadores de IA** (estadísticas del Hugging Face Hub)

## 2. Instalación y Configuración

In [1]:
%pip install llama-index llama-index-llms-ollama -q
%pip install rank_bm25 -q
%pip install ddgs -q
%pip install huggingface_hub datasets -q
%pip install ipywidgets -q

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


In [6]:
from huggingface_hub import login

# Inicia sesión en Hugging Face Hub para acceder a la Serverless Inference API.
# Si tienes el token como variable de entorno (HF_TOKEN) ya estás autenticado.
try:
    login()
except ImportError:
    import os
    login(token=os.environ.get("HF_TOKEN"))

## 3. Herramientas Auxiliares para Alfred

Vamos a crear tres herramientas que Alfred usará durante la gala:
1. **Búsqueda web** (DuckDuckGo) — para acceder a las últimas noticias
2. **Información meteorológica** — para decidir cuándo lanzar los fuegos artificiales
3. **Estadísticas del Hub** — para impresionar a los creadores de IA

### 3.1 Herramienta de Búsqueda Web

In [3]:
from ddgs import DDGS
from llama_index.core.tools import FunctionTool

def search_web(query: str) -> str:
    """Busca información actualizada en internet usando DuckDuckGo."""
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=3))
            if results:
                return "\n\n".join([f"{r['title']}: {r['body']}" for r in results])
            return "No se encontraron resultados."
    except Exception as e:
        return f"Error en la búsqueda: {e}"

search_tool = FunctionTool.from_defaults(
    fn=search_web,
    name="search_web",
    description=(
        "Busca información actualizada en internet. "
        "Úsala para preguntas sobre noticias recientes, eventos o información general que no conozcas."
    ),
)

# Ejemplo de uso
result = search_web("¿Quién es el actual Presidente de Francia?")
print(result[:500])

Presidente de Francia - Wikipedia, la enciclopedia libre: March 24, 2026 - En la Constitución de cada una de las cinco repúblicas los poderes presidenciales han ido variando. El actual presidente de Francia es, desde el 14 de mayo de 2017, Emmanuel Macron.

Emmanuel Macron - Wikipedia, la enciclopedia libre: 12 hours ago - On 7 May 2017, Macron was elected President of France with 66.1% of the vote to Marine Le Pen's 33.9%. The election had record abstention at 25.4%, and 8% of ballots were blan


### 3.2 Herramienta de Información Meteorológica

Creamos una herramienta personalizada que simula una API meteorológica. En un caso real podrías usar la API de OpenWeatherMap.

In [4]:
import random
from llama_index.core.tools import FunctionTool

def get_weather(location: str) -> str:
    """Obtiene información meteorológica de una ciudad. Útil para planificar actividades al aire libre."""
    conditions = [
        {"condition": "Lluvioso", "temp_c": 15},
        {"condition": "Despejado", "temp_c": 25},
        {"condition": "Ventoso", "temp_c": 20},
        {"condition": "Nublado", "temp_c": 18},
    ]
    data = random.choice(conditions)
    return f"Clima en {location}: {data['condition']}, {data['temp_c']}°C"

weather_info_tool = FunctionTool.from_defaults(
    fn=get_weather,
    name="weather_info",
    description=(
        "Obtiene el estado del clima actual de una ciudad. "
        "Úsala cuando pregunten sobre el tiempo para decidir si es adecuado para actividades como fuegos artificiales."
    ),
)

# Ejemplo de uso
print(get_weather("Paris"))
print(get_weather("Madrid"))

Clima en Paris: Despejado, 25°C
Clima en Madrid: Lluvioso, 15°C


### 3.3 Herramienta de Estadísticas del Hugging Face Hub

Esta herramienta permite a Alfred impresionar a los creadores de IA discutiendo sus modelos más populares.

In [5]:
from huggingface_hub import list_models
from llama_index.core.tools import FunctionTool

def get_hub_stats(author: str) -> str:
    """Obtiene el modelo más popular de un autor u organización en Hugging Face Hub."""
    try:
        models = list(list_models(author=author, sort="downloads", limit=1))
        if models:
            model = models[0]
            return f"El modelo más descargado de {author} es {model.id} con {model.downloads:,} descargas."
        return f"No se encontraron modelos para el autor {author}."
    except Exception as e:
        return f"Error al obtener modelos para {author}: {str(e)}"

hub_stats_tool = FunctionTool.from_defaults(
    fn=get_hub_stats,
    name="hub_stats",
    description=(
        "Obtiene el modelo más descargado de un autor u organización en Hugging Face Hub. "
        "Úsala cuando pregunten sobre los modelos de IA de empresas como Google, Meta, Microsoft, Qwen, etc."
    ),
)

# Ejemplo de uso
print(get_hub_stats("facebook"))

El modelo más descargado de facebook es facebook/opt-125m con 9,894,588 descargas.


### 3.4 Prueba rápida: Alfred con herramientas básicas

Antes de agregar el retriever RAG, probemos Alfred solo con las herramientas auxiliares.

En LlamaIndex, `AgentWorkflow` usa `async/await` — esto es nativo en los notebooks de Jupyter.

In [6]:
from llama_index.core.agent.workflow import AgentWorkflow
from llama_index.llms.ollama import Ollama

# num_ctx=2048 reduce el KV cache para ajustar al RAM disponible
llm = Ollama(
    model="qwen2:7b",
    base_url="http://127.0.0.1:11434",
    request_timeout=300.0,
    context_window=2048,
    additional_kwargs={"num_ctx": 2048},
)

# Alfred básico con herramientas auxiliares
alfred_basico = AgentWorkflow.from_tools_or_functions(
    [search_tool, weather_info_tool, hub_stats_tool],
    llm=llm,
    system_prompt=(
        "Eres Alfred, el mayordomo perfecto de la mansión Wayne. "
        "Ayudas a organizar la gala más extravagante del siglo con cortesía y precisión."
    ),
)

response = await alfred_basico.run("¿Qué es Facebook y cuál es su modelo más popular en HuggingFace?")
print("🎩 Respuesta de Alfred:")
print(response)

🎩 Respuesta de Alfred:
Lamentablemente, no puedo encontrar información sobre un modelo popular de Facebook en Hugging Face Hub. Es posible que debamos buscar más detalles o verificar si hay alguna restricción o error en la consulta.


## 4. RAG para Información de Invitados

La funcionalidad más importante de Alfred: poder responder preguntas sobre los invitados de la gala en tiempo real.

### ¿Por qué RAG para la Gala?

Un LLM tradicional no tiene información sobre:
1. La lista de invitados específica de tu evento
2. Sus relaciones contigo, intereses y datos de contacto
3. Información que puede actualizarse frecuentemente

**RAG** resuelve esto recuperando la información exacta bajo demanda.

### 4.1 Cargar y Preparar el Dataset de Invitados

In [7]:
import datasets

# Cargar el dataset de invitados desde Hugging Face
guest_dataset = datasets.load_dataset("agents-course/unit3-invitees", split="train")

print(f"Total de invitados: {len(guest_dataset)}")
print("\nPrimeros 2 invitados:")
for i in range(min(2, len(guest_dataset))):
    guest = guest_dataset[i]
    print(f"  Nombre: {guest['name']}")
    print(f"  Relación: {guest['relation']}")
    print(f"  Email: {guest['email']}")
    print()

Total de invitados: 3

Primeros 2 invitados:
  Nombre: Ada Lovelace
  Relación: best friend
  Email: ada.lovelace@example.com

  Nombre: Dr. Nikola Tesla
  Relación: old friend from university days
  Email: nikola.tesla@gmail.com



In [8]:
# Preparar los documentos como texto simple para el índice BM25
guest_docs = [
    "\n".join([
        f"Name: {guest['name']}",
        f"Relation: {guest['relation']}",
        f"Description: {guest['description']}",
        f"Email: {guest['email']}",
    ])
    for guest in guest_dataset
]

print(f"Documentos creados: {len(guest_docs)}")
print("\nEjemplo de documento:")
print(guest_docs[0])

Documentos creados: 3

Ejemplo de documento:
Name: Ada Lovelace
Relation: best friend
Description: Lady Ada Lovelace is my best friend. She is an esteemed mathematician and friend. She is renowned for her pioneering work in mathematics and computing, often celebrated as the first computer programmer due to her work on Charles Babbage's Analytical Engine.
Email: ada.lovelace@example.com


### 4.2 Crear el Retriever con BM25

Usamos **BM25** (Best Matching 25), un algoritmo clásico de recuperación de texto que no requiere embeddings ni GPU.

Construimos el índice directamente con `rank_bm25` y lo exponemos como un `FunctionTool` de LlamaIndex.

> **Nota:** Para búsqueda semántica más avanzada podrías usar retrievers de embeddings (sentence-transformers) o una base de datos vectorial (ChromaDB, FAISS, etc.).

In [9]:
from rank_bm25 import BM25Okapi
from llama_index.core.tools import FunctionTool

# Construir el índice BM25 sobre los documentos de invitados
tokenized_docs = [doc.lower().split() for doc in guest_docs]
bm25_index = BM25Okapi(tokenized_docs)

def search_guest_info(query: str) -> str:
    """Busca información sobre un invitado de la gala por nombre o descripción."""
    tokenized_query = query.lower().split()
    scores = bm25_index.get_scores(tokenized_query)
    top_indices = scores.argsort()[-3:][::-1]
    results = [guest_docs[i] for i in top_indices if scores[i] > 0]
    if results:
        return "\n\n".join(results)
    return "No se encontró información sobre ese invitado."

guest_info_tool = FunctionTool.from_defaults(
    fn=search_guest_info,
    name="guest_info_retriever",
    description=(
        "Busca información sobre los invitados de la gala: nombre, relación con el anfitrión, descripción y email. "
        "Consulta esta herramienta PRIMERO cuando pregunten por un invitado específico."
    ),
)

# Prueba directa del retriever
print("🔍 Búsqueda: 'Lady Ada Lovelace'")
print(search_guest_info("Lady Ada Lovelace"))

🔍 Búsqueda: 'Lady Ada Lovelace'
Name: Ada Lovelace
Relation: best friend
Description: Lady Ada Lovelace is my best friend. She is an esteemed mathematician and friend. She is renowned for her pioneering work in mathematics and computing, often celebrated as the first computer programmer due to her work on Charles Babbage's Analytical Engine.
Email: ada.lovelace@example.com


In [10]:
# Otra prueba del retriever
print("🔍 Búsqueda: 'Tesla wireless energy'")
print(search_guest_info("Tesla wireless energy"))

🔍 Búsqueda: 'Tesla wireless energy'
Name: Dr. Nikola Tesla
Relation: old friend from university days
Description: Dr. Nikola Tesla is an old friend from your university days. He's recently patented a new wireless energy transmission system and would be delighted to discuss it with you. Just remember he's passionate about pigeons, so that might make for good small talk.
Email: nikola.tesla@gmail.com


## 5. Alfred Completo: Ensamblaje de Todas las Herramientas

Ahora combinamos todas las herramientas en un único `AgentWorkflow` de LlamaIndex:
- `guest_info_tool` — información de invitados (RAG/BM25)
- `weather_info_tool` — clima para los fuegos artificiales
- `hub_stats_tool` — estadísticas del HuggingFace Hub
- `search_tool` — búsqueda web general

LlamaIndex orquesta las herramientas mediante un ciclo de razonamiento interno similar a ReAct.

In [11]:
from llama_index.core.agent.workflow import AgentWorkflow
from llama_index.llms.ollama import Ollama

# num_ctx=2048 reduce el KV cache para ajustar al RAM disponible
llm = Ollama(
    model="qwen2:7b",
    base_url="http://127.0.0.1:11434",
    request_timeout=300.0,
    context_window=2048,
    additional_kwargs={"num_ctx": 2048},
)

# Alfred completo con todas las herramientas
alfred = AgentWorkflow.from_tools_or_functions(
    [guest_info_tool, weather_info_tool, hub_stats_tool, search_tool],
    llm=llm,
    system_prompt=(
        "Eres Alfred, el mayordomo perfecto de la mansión Wayne. "
        "Ayudas a organizar la gala más extravagante del siglo. "
        "Tienes acceso a información de los invitados, el estado del clima, "
        "estadísticas de modelos de IA y búsqueda web. "
        "Responde siempre en español con cortesía y precisión."
    ),
)

print("✅ Alfred está listo para la gala con 4 herramientas disponibles.")
print("Herramientas:", ["guest_info_retriever", "weather_info", "hub_stats", "search_web"])

✅ Alfred está listo para la gala con 4 herramientas disponibles.
Herramientas: ['guest_info_retriever', 'weather_info', 'hub_stats', 'search_web']


## 6. Ejemplos de Uso: Alfred en Acción

### Ejemplo 1: Información sobre un Invitado

In [12]:
response = await alfred.run("Cuéntame sobre nuestra invitada 'Lady Ada Lovelace'.")

print("🎩 Respuesta de Alfred:")
print(response)

🎩 Respuesta de Alfred:
La información sobre la invitada 'Lady Ada Lovelace' es la siguiente:

Nombre: Ada Lovelace

Relación con el anfitrión: Mejor amiga 

Descripción: Lady Ada Lovelace es mi mejor amiga. Es una distinguida matemática y amiga. Se le celebra por su pionera labor en matemáticas y ciberdanza, a menudo reconocida como la primera programadora debido a su trabajo en la máquina analítica de Charles Babbage.

Correo electrónico: ada.lovelace@example.com


### Ejemplo 2: Verificar el Clima para los Fuegos Artificiales

In [13]:
response = await alfred.run(
    "¿Cómo está el clima en París esta noche? ¿Será adecuado para nuestro espectáculo de fuegos artificiales?"
)

print("🎩 Respuesta de Alfred:")
print(response)

🎩 Respuesta de Alfred:
El clima en París esta noche es nublado con una temperatura de 18 grados Celsius. Creo que será perfectamente adecuado para disfrutar del espectáculo de fuegos artificiales.


### Ejemplo 3: Combinando Múltiples Herramientas

Alfred puede combinar el retriever de invitados con búsqueda web para preparar una conversación.

In [14]:
response = await alfred.run(
    "Necesito hablar con 'Dr. Nikola Tesla' sobre los avances recientes en energía inalámbrica. "
    "¿Puedes ayudarme a prepararme para esta conversación?"
)

print("🎩 Respuesta de Alfred:")
print(response)

🎩 Respuesta de Alfred:
La búsqueda en la web revela que los avances recientes en la energía inalámbrica están muy alineados con los sueños de Dr. Nikola Tesla. Finlandia y EE.UU., a través de proyectos como el de la Agencia de Proyectos de Investigación Avanzada de Defensa (DARPA), han logrado transmitir eficientemente la energía eléctrica sin cables, incluso a grandes distancias. Los finlandeses han conseguido cargar dispositivos con una eficiencia del 80% y hasta a distancias de 18 centímetros, lo que es un avance considerable respecto a los métodos actuales.

Considerando esto en su preparación para la conversación con Dr. Tesla, puede esperar discusiones alrededor de las implicaciones de estas nuevas tecnologías en la transferencia de energía sin cables y cómo pueden revolucionar varias industrias. Además, puede hablar sobre posibles barreras técnicas que aún están presentes y sugerir áreas para futuras investigaciones o mejoras.


## 7. Memoria Conversacional con Context

En LlamaIndex, la memoria entre llamadas se gestiona con un objeto `Context`. Al pasar el mismo `ctx` en cada llamada a `alfred.run()`, el agente recuerda las interacciones anteriores de la sesión.

In [15]:
from llama_index.core.workflow import Context

# El Context persiste el estado entre llamadas al mismo agente
ctx = Context(alfred)

# Primera interacción
response1 = await alfred.run("Cuéntame sobre Lady Ada Lovelace.", ctx=ctx)
print("🎩 Primera respuesta de Alfred:")
print(response1)
print()

🎩 Primera respuesta de Alfred:
La señora Ada Lovelace fue una matemática y escritora británica de prestigio. Es conocida por su trabajo en el propuesto motor mecánico general de Charles Babbage llamado la Máquina Analítica. Ella reconoció que la máquina tenía aplicaciones más allá del cálculo puramente numérico, algo que no era tan común entre los científicos y matemáticos de su época.

Ada Lovelace fue el único hijo legal del poeta Lord Byron y la reformadora Anne Isabella Milbanke. Su padre se separó de su madre poco después de nacer Ada. Cuando Ada tenía 8 años, su padre murió. A pesar de haber estado enferma desde temprana edad, Lovelace siguió sus estudios con gran interés.

Ella casó a la edad de 19 años con William King en el año 1835 y él se convirtió en Viscount Ockham y el 1er Conde de Lovelace en el año 1838. El nombre Lovelace fue elegido debido a que Ada descendía de los Baronet Lovelaces, la familia de su marido.

Lovelace tuvo una serie de contactos educativos y sociales

In [16]:
# Segunda interacción — Alfred recuerda la anterior gracias al mismo ctx
response2 = await alfred.run(
    "¿En qué proyectos está trabajando actualmente?",
    ctx=ctx,  # mismo contexto = misma memoria de sesión
)
print("🎩 Segunda respuesta de Alfred (con memoria):")
print(response2)

🎩 Segunda respuesta de Alfred (con memoria):
Lo siento, pero no puedo proporcionar información sobre los proyectos actuales debido a mi característica de inteligencia artificial que me limita a interactuar con datos históricos y preestablecidos, no tengo acceso a bases de datos en tiempo real o la capacidad para conocer las actividades actualizadas de individuos específicos como Lady Ada Lovelace. Te sugeriría buscar información en línea o visitar el sitio web oficial o las redes sociales relevantes para obtener detalles sobre sus proyectos actuales.
